# Wavebit Tutorial 1
 Modeling and Electromagnetic simulation

In [ ]:

import matplotlib.pyplot as plt
import numpy as np

import wavebit

In [ ]:
tetm = 'te'  # 'te' or 'tm'. A declaration: this problem has no polarization
             # and no azimuth degree of freedom, which is what buys the fast
             # 1D path. For an angle between them, use psi (Case 2).
psi = 0.25 * np.pi  # the linear polarization angle, for the general cases below

top = 1.0  # superstrate: relative permittivity. 1.0 is vacuum
bot = 1.0  # substrate, same reading

theta = 20 * np.pi / 180
phi = 50 * np.pi / 180

wavelength = 900

thickness = [500]
period = [1000]

truncation_order = [30]

complex_dtype = np.complex128


## 1.1 Modeling

### structure (the layer stack)

`structure` is the layer stack. Here it is a **raster**: a 3-dimensional array whose elements are **relative permittivity**. Complex values are supported (`Im(eps) > 0` is loss, in this library's `e^{-iwt}` convention), as are 2-dimensional gratings and multi-layered structures.

The dimensions are `(layer, y, x)`. Layers are listed **bottom to top**:

```text
          +z ↑
top       above z = d0 + d1
          ┌────────────────────┐
layer 1   │ structure[1] · d1  │
          ├────────────────────┤
layer 0   │ structure[0] · d0  │
z = 0     └────────────────────┘
bot       below z = 0
```

* Axis 0 selects a layer; `thickness[i]` and `mu[i]` belong to the same layer.
* Axis 1 samples y (or fractional v). Row 0 is the negative-y edge in a rectangular cell.
* Axis 2 samples x (or fractional u). Column 0 is the negative-x edge.
* The xy origin is the cell center. The frame is right-handed and +z points upward.

**One number per pixel is what makes the raster a raster.** It is a *sampled field*, not an
assignment, which is exactly why topology optimization reaches for it (Tutorial 2). The other
spelling of `structure` -- the parametric shape dicts -- names a `wavebit.Material` per region
instead, and there a bare number is the shorthand for `wavebit.Material(epsilon=x)`. A magnetic
raster passes a second array of the same shape as `mu=`; on the shape path the permeability
rides inside each `Material`.

If what you have is a refractive index, convert it on the way in: `wavebit.eps_from_index(n)`
is `n**2`, and `wavebit.index_from_eps(eps)` goes back. The conversion is worth writing out,
because an index left unconverted is a perfectly legal permittivity -- it solves, it conserves
energy, and it answers a different question.


#### structure examples

In [ ]:
structure_1d_s = np.array([
    [
        [0, 1, 0, 1, 1, 0, 1, 0, 1, 1],
    ],
]) * 24 + 1  # relative permittivity: 1 (vacuum) and 25

1D grating, single layer

In [ ]:
structure_1d_m = np.array([
    [
        [0, 0, 0, 0, 0, 0, 1, 0, 1, 1],
    ],
    [
        [1, 1, 1, 1, 1, 0, 1, 0, 1, 0],
    ],
    [
        [0, 1, 0, 1, 1, 0, 1, 0, 1, 1],
    ],
]) * 24 + 1  # relative permittivity: 1 (vacuum) and 25

1D grating, multi layer

In [ ]:
structure_2d_s = np.array([
    [
        [0, 1, 0, 1, 1, 0, 1, 0, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 0, 1, 1],
    ],
]) * 24 + 1  # relative permittivity: 1 (vacuum) and 25

2D grating, single layer

In [ ]:
structure_2d_m = wavebit.eps_from_index(np.array([
    [
        [0, 1, 0, 1, 1, 1, 1, 0, 1, 1],
        [0, 1, 3, 1, 1, 1, 1, 1, 1, 1],
    ],
    [
        [0, 1, 0, 1, 1, 0, 1, 0, 1, 1],
        [1, 1, 1, 2, 0, 1, 2, 0, 1, 1],
    ],
    [
        [1, 1, 0, 1, 1, 0, 1, 0, 2, 1],
        [0, 1, 1, 1, 2, 4, 1, 0, 1, 1],
    ],
    [
        [0, 1, 0, 1, 1, 0, 1, 0, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 0, 1, 1],
    ],
]) * 4 + 1)  # a pattern stated as refractive index, squared once here

2D grating, multi layer

### Wave

#### $\theta$: angle of incidence


          Incidence   normal
               \\      |
                \\     |
                 \\ 𝜃 (angle of incidence)     
                  \\   |
                   \\  |             
                    \\ |
                     \\|                       Z-axis
     ____________________________________        |
    |              Layer 1               |       |
    |____________________________________|       |_____ X-axis

#### $\phi$: azimuthal (rotation) angle

    



            plane of incidence
              |     \
              |      \
              |       \
              |        \
              |         \
              |          \
              |           \
               \           \
           /    \           \                 /
          /      \           \               / 
         / -------\---------- \             /              
        /          \ 𝜙 (rotation angle)    /  /          Z-axis  
       /            \          |          /  /            |
      /______________\_________|_________/  / /           |
      |   Layer 1     \        |         | / /            |_____ X-axis
      |________________\_______|_________|/ /            /
      |   Layer 2       \      |         | /            /
      |__________________\_____|_________|/           Y-axis
                          \    |                     
                   



  In 2D (projection on XY-plane),
  
     |
     |
     |--------------- X-axis
     |\\ 𝜙 (rotation angle)
     | \\
     |  \\
     |   \\
     |    \\
     |     \\
     Y     Incidence
    axis
    

## 1.2 EM Simulation

In [ ]:
solver = wavebit.rcwa(backend='numpy', 
                     tetm=tetm, top=top, bot=bot, theta=theta, truncation_order=truncation_order, 
                     wavelength=wavelength, period=period, structure=structure_1d_s, thickness=thickness,
                       complex_dtype=complex_dtype)

call the Wavebit operator by `wavebit.rcwa`.
Here, backend can be selected with keyword `backend`

```python
backend = 'numpy'  # Numpy backend
backend = 'jax'  # JAX backend
backend = 'torch'  # PyTorch backend
```


### Case 1: 1D TE/TM
For 1D TE or TM case, fast calculation can be achieved.
Setting phi as `None` (which is default). 

This returns result of either TE or TM while the general case does both.

In [ ]:
solver = wavebit.rcwa(backend='numpy', tetm=tetm, top=top, bot=bot, theta=theta, truncation_order=truncation_order, 
                     wavelength=wavelength, period=period, structure=structure_1d_s, thickness=thickness,
                       complex_dtype=complex_dtype)
solver.truncation_order = [200]
solver.tetm = 'te'  # 'te' or 'tm'. Anything between them is an angle: see Case 2.
solver.phi = None  # None is by default. This explicit assign is for demo.

result = solver.solve()

For 1D TE/TM, the result is in `res` in `result`.

In [ ]:
res = result.incident
R = res.R
T = res.T
print('Diffraction efficiency: ', R.sum(), T.sum())


### Case 2: General
General form includes 1D grating with non TE or TM input, 1D conical and 2D gratings.

This returns 3 results: result from given polarization(or psi), result from pure-s incidence, and result from pure-p incidence.

In [ ]:
solver = wavebit.rcwa(backend='numpy', psi=psi, top=top, bot=bot, theta=theta, phi=phi, truncation_order=truncation_order, 
                     wavelength=wavelength, period=period, structure=structure_1d_s, thickness=thickness,
                       complex_dtype=complex_dtype)
solver.truncation_order = [200]
solver.psi = np.pi / 4  # the angle a fractional pol used to mean
solver.phi = 30

result = solver.solve()


The return `result` has 3 sub-result classes each of which contains: result for given polarization (or psi), for pure-s incidence and for pure-p incidence.

In [ ]:
res = result.incident
s_inc = result.s_inc
p_inc = result.p_inc

R, T = res.R, res.T
R_s, T_s = s_inc.R, s_inc.T
R_p, T_p = p_inc.R, p_inc.T

print('Diffraction efficiency: ', R.sum(), T.sum())
print('Diffraction efficiency: ', R_s.sum(), T_s.sum())
print('Diffraction efficiency: ', R_p.sum(), T_p.sum())


call the Wavebit operator by `wavebit.rcwa`.
Here, backend can be selected with keyword `backend`

```python
backend = 'numpy'  # Numpy backend
backend = 'jax'  # JAX backend
backend = 'torch'  # PyTorch backend
```


## 1.3 RCWA Result

In [ ]:
truncation_order = dict(x=4, y=2)
thickness = [245, 400, 200, 100]
period = dict(x=1000, y=2000)

structure_2d_m = wavebit.eps_from_index(np.array([
    [
        [0, 1, 0, 1, 1, 1, 1, 0, 1, 1],
        [0, 1, 3, 1, 1, 1, 1, 1, 1, 1],
    ],
    [
        [0, 1, 0, 1, 1, 0, 1, 0, 1, 1],
        [1, 1, 1, 2, 0, 1, 2, 0, 1, 1],
    ],
    [
        [1, 1, 0, 1, 1, 0, 1, 0, 2, 1],
        [0, 1, 1, 1, 2, 4, 1, 0, 1, 1],
    ],
    [
        [0, 1, 0, 1, 1, 0, 1, 0, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 0, 1, 1],
    ],
]) * 4 + 1)  # a pattern stated as refractive index, squared once here


In [ ]:
solver = wavebit.rcwa(backend='numpy', psi=psi, top=top, bot=bot, theta=theta, phi=phi,
                      truncation_order=truncation_order, wavelength=wavelength, period=period, structure=structure_2d_m, 
                      thickness=thickness, complex_dtype=complex_dtype)
solver.psi = np.pi / 4  # the angle a fractional pol used to mean

result = solver.solve()

res = result.incident
s_inc = result.s_inc
p_inc = result.p_inc

### RCWA result

In [ ]:
attrs = vars(res)
print(', '.join("%s" % item for item in attrs))


Attributes of subresult instance, in the order printed above. The naming rule is
**uppercase is power, lowercase is amplitude**; the `_s` / `_p` suffix names the
**outgoing** polarization, not the incidence.


r_s: complex Fresnel reflection amplitude, s component <br>
r_p: complex Fresnel reflection amplitude, p component <br>
t_s: complex Fresnel transmission amplitude, s component <br>
t_p: complex Fresnel transmission amplitude, p component <br>
R: diffraction efficiency of reflection in total <br>
R_s: diffraction efficiency of reflection into the s component <br>
R_p: diffraction efficiency of reflection into the p component <br>
T: diffraction efficiency of transmission in total <br>
T_s: diffraction efficiency of transmission into the s component <br>
T_p: diffraction efficiency of transmission into the p component <br>



### Diffraction Efficiency

For top illumination, light enters from `top`, travels toward −z and leaves through
`bot`. Reflected orders return to `top`; transmitted orders leave through `bot`.
The top finite layer is `structure[-1]`, the bottom one `structure[0]`.
With `incidence_side="bottom"`, R and T follow the illumination side while the
geometry, layer order and +z axis remain fixed.


## 1.4 Field construction result

### 1D TE

In [ ]:
tetm = 'te'  # 'te' or 'tm'. A declaration: this problem has no polarization
             # and no azimuth degree of freedom, which is what buys the fast
             # 1D path. For an angle between them, use psi (Case 2).
psi = 0.25 * np.pi  # the linear polarization angle, for the general cases below

top = 1.0  # superstrate: relative permittivity. 1.0 is vacuum
bot = 1.0  # substrate, same reading

theta = 20 * np.pi / 180
phi = 50 * np.pi / 180

wavelength = 900

thickness = [500]
period = [1000]

truncation_order = [30]

complex_dtype = np.complex128

solver = wavebit.rcwa(backend='numpy', tetm=tetm, top=top, bot=bot, theta=theta, truncation_order=truncation_order, 
                     wavelength=wavelength, period=period, structure=structure_1d_s, thickness=thickness,
                       complex_dtype=complex_dtype, answers='field')

result, field_cell = solver.solve_field(sample_z=100, sample_y=1, sample_x=100)

ZX direction (Side View)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10,2))
if tetm == 'te':  # TE
    title = ['1D Ey', '1D Hx', '1D Hz', ]
else:  # TM
    title = ['1D Hy', '1D Ex', '1D Ez', ]

for ix in range(len(title)):
    val = abs(field_cell[:, 0, :, ix]) ** 2
    im = axes[ix].imshow(val, cmap='jet', aspect='auto', origin='lower')
    # plt.clim(0, 2)  # identical to caxis([-4,4]) in MATLAB
    fig.colorbar(im, ax=axes[ix], shrink=1)
    axes[ix].title.set_text(title[ix])

plt.show()

### 1D TM

In [ ]:
tetm = 'tm'  # 'te' or 'tm'

top = 1.0  # superstrate: relative permittivity. 1.0 is vacuum
bot = 1.0  # substrate, same reading

theta = 20 * np.pi / 180
phi = 50 * np.pi / 180

wavelength = 900

thickness = [500]
period = [1000]

truncation_order = [30]

complex_dtype = np.complex128

solver = wavebit.rcwa(backend='numpy', tetm=tetm, top=top, bot=bot, theta=theta, truncation_order=truncation_order, 
                     wavelength=wavelength, period=period, structure=structure_1d_s, thickness=thickness,
                       complex_dtype=complex_dtype, answers='field')

result, field_cell = solver.solve_field(sample_z=100, sample_y=1, sample_x=100)

ZX direction (Side View)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10,2))
if tetm == 'te':  # TE
    title = ['1D Ey', '1D Hx', '1D Hz', ]
else:  # TM
    title = ['1D Hy', '1D Ex', '1D Ez', ]

for ix in range(len(title)):
    val = abs(field_cell[:, 0, :, ix]) ** 2
    im = axes[ix].imshow(val, cmap='jet', aspect='auto', origin='lower')
    # plt.clim(0, 2)  # identical to caxis([-4,4]) in MATLAB
    fig.colorbar(im, ax=axes[ix], shrink=1)
    axes[ix].title.set_text(title[ix])

plt.show()

### General
In general cases, the fields can be individually simulated with 3 different polarized incidence simultaneously (with additional calculation time) as RCWA does.


In [ ]:
truncation_order = dict(x=4, y=2)
thickness = [245, 400, 200, 100]
period = dict(x=1000, y=2000)

structure_2d_m = wavebit.eps_from_index(np.array([
    [
        [0, 1, 0, 1, 1, 1, 1, 0, 1, 1],
        [0, 1, 3, 1, 1, 1, 1, 1, 1, 1],
    ],
    [
        [0, 1, 0, 1, 1, 0, 1, 0, 1, 1],
        [1, 1, 1, 2, 0, 1, 2, 0, 1, 1],
    ],
    [
        [1, 1, 0, 1, 1, 0, 1, 0, 2, 1],
        [0, 1, 1, 1, 2, 4, 1, 0, 1, 1],
    ],
    [
        [0, 1, 0, 1, 1, 0, 1, 0, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 0, 1, 1],
    ],
]) * 4 + 1)  # a pattern stated as refractive index, squared once here

solver = wavebit.rcwa(backend='numpy', psi=psi, top=top, bot=bot, theta=theta, phi=phi,
                      truncation_order=truncation_order, wavelength=wavelength, period=period, structure=structure_2d_m, 
                      thickness=thickness, complex_dtype=complex_dtype,
                      answers='field')


In [ ]:
result, field_cell = solver.solve_field(sample_z=100, sample_y=100, sample_x=100, incidences=('incident', 's', 'p'))


`incidences` names which incidences to recover the field for — one leading-axis entry each, in the order you ask for them:

* `'incident'` — the incidence you set (`psi`, or the `polarization` pair)
* `'s'` — pure s-polarized (TE) incidence, which the coupled solve produces for free
* `'p'` — pure p-polarized (TM) incidence, likewise

These are the same three the result object calls `incident`, `s_inc` and `p_inc`.

In [ ]:
field_cell.shape

The first axis is 3 where each subarrays are the set of fields generated from user set pol, TE and TM cases.

The second axis is Z-direction.

The third axis is Y-direction

The fourth is X-direction.

The last represents the type of the field (Ex, Ey, Ez, Hx, Hy, Hz)

ZX direction (Side View)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 2))
title = ['2D Ex', '2D Ey', '2D Ez', '2D Hx', '2D Hy', '2D Hz', ]

for ix in range(3):
    val = abs(field_cell[0, :, 0, :, ix]) ** 2
    im = axes[ix].imshow(val, cmap='jet', aspect='auto', origin='lower')
    # plt.clim(0, 2)  # identical to caxis([-4,4]) in MATLAB
    fig.colorbar(im, ax=axes[ix], shrink=1)
    axes[ix].title.set_text(title[ix])

plt.show()

fig, axes = plt.subplots(1, 3, figsize=(10, 2))
for ix in range(3, 6, 1):
    val = abs(field_cell[0, :, 0, :, ix]) ** 2
    im = axes[ix-3].imshow(val, cmap='jet', aspect='auto', origin='lower')
    # plt.clim(0, 2)  # identical to caxis([-4,4]) in MATLAB
    fig.colorbar(im, ax=axes[ix-3], shrink=1)
    axes[ix-3].title.set_text(title[ix])

plt.show()

#### YZ Direction (Top View)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 2))
title = ['2D Ex', '2D Ey', '2D Ez', '2D Hx', '2D Hy', '2D Hz', ]

for ix in range(3):
    val = abs(field_cell[0, 0, :, :, ix]) ** 2
    im = axes[ix].imshow(val, cmap='jet', aspect='auto', origin='lower')
    # plt.clim(0, 2)  # identical to caxis([-4,4]) in MATLAB
    fig.colorbar(im, ax=axes[ix], shrink=1)
    axes[ix].title.set_text(title[ix])

plt.show()

fig, axes = plt.subplots(1, 3, figsize=(10, 2))
for ix in range(3, 6, 1):
    val = abs(field_cell[0, 0, :, :, ix]) ** 2
    im = axes[ix-3].imshow(val, cmap='jet', aspect='auto', origin='lower')
    # plt.clim(0, 2)  # identical to caxis([-4,4]) in MATLAB
    fig.colorbar(im, ax=axes[ix-3], shrink=1)
    axes[ix-3].title.set_text(title[ix])

plt.show()